# Part 4 - Drift and Observability Using One UWB Beacon

This notebook demonstrates relative pose estimation with a single UWB anchor, fusing noisy odometry (VIO) with UWB range measurements using an SE(2) Extended Kalman Filter (EKF). We explore drift propagation and observability analysis in this configuration.

In [ ]:
# Cell 2: Imports

import numpy as np
import matplotlib.pyplot as plt

from manifpy import SE2

import sys
sys.path.insert(0, '/workspace/Python/Relative-localisation-tutorial/src')

from config import TrajectoryConfig, VIOConfig, UWBNoiseConfig, EKFConfig
from trajectory import generate_trajectory, sgal3_to_se2_poses
from vio_sensor import simulate_vio_odometry
from uwb import simulate_uwb_ranges, plot_anchor_positions, simulate_uwb_bearings
from plotting import plot_trajectory_2d, plot_error_ellipse
from ekf import simulate_ekf, simulate_ekf_with_bearing

In [ ]:
# Cell 3: Trajectory generation

# Parameters for trajectory generation of robot 1
traj_config = TrajectoryConfig(
    duration=100.0,
    timestep=0.05,   # 20 Hz odometry rate
    seed=42          # Reproducibility
)

# Generate trajectory
trajectory = generate_trajectory(
    duration=traj_config.duration,
    timestep=traj_config.timestep,
    seed=traj_config.seed
)

# Convert to SE(2) poses for visualization
poses_gt = sgal3_to_se2_poses(trajectory.X_RRi)

# Visualize ground truth trajectory
fig, ax = plt.subplots(figsize=(10, 8))
plot_trajectory_2d(ax, poses_gt, label='Ground Truth', color='k')
ax.set_title('Robot Trajectory (Perfect)')
plt.show()

In [ ]:
# Cell 4a: Exercise 1 - Robot to Anchor Relative Pose Estimation

# Configuration
anchor_pos = np.array([5.0, 5.0])  # Single anchor position

# Noise configurations
vio_cfg = VIOConfig(sig_v=0.1, sig_w=0.05)
uwb_cfg = UWBNoiseConfig(sigma_range=0.1)  # 10 cm range noise
ekf_cfg = EKFConfig()

# Simulate noisy odometry and UWB ranges
poses_noisy, odometry_traj = simulate_vio_odometry(trajectory, vio_cfg)
ranges = simulate_uwb_ranges(
    poses_noisy,
    np.array([anchor_pos]),
    uwb_cfg
)

# Run EKF with mixed-rate updates (odometry at 20 Hz, UWB at 10 Hz)
state_seq, cov_seq, t_seq = simulate_ekf(
    trajectory,
    anchor_pos,
    ekf_cfg,
    (vio_cfg, uwb_cfg),
    seed=42
)

# Convert to arrays for plotting
poses_est = np.array(state_seq)

# Plot results with error ellipses
fig, ax = plt.subplots(figsize=(10, 8))
plot_trajectory_2d(ax, poses_gt, label='Ground Truth', color='k')
plot_trajectory_2d(ax, poses_est, label='EKF Estimate', color='b', linestyle='--')
plot_anchor_positions(ax, np.array([anchor_pos]))

# Add error ellipses (every 10th sample for visibility)
plot_every = 10
for i in range(0, len(poses_est), plot_every):
    plot_error_ellipse(ax, poses_est[i], cov_seq[i], color='blue', n_std=2.0)

ax.set_title('Exercise 1: Robot to Anchor Relative Pose Estimation')
ax.legend()
plt.show()

In [ ]:
# Cell 4b: Exercise 2 - Anchor to Robot Relative Pose Estimation (Placeholder)

# TODO: Implement anchor-to-robot re-estimation with separate EKF state vector
# This exercise will demonstrate how observability affects the solution path

print("Exercise 2: Placeholder for future implementation")
print("Topic: Anchor to robot relative pose estimation (re-computation approach)")

# Cell 5: Observability Analysis (Placeholder)

In [ ]:
# Cell 6: Exercise 3 - Range and Bearing Fusion

# Configuration (same as before but with bearing noise)
anchor_pos = np.array([5.0, 5.0])
vio_cfg = VIOConfig(sig_v=0.1, sig_w=0.05)
uwb_cfg = UWBNoiseConfig(sigma_range=0.1, sigma_bearing=0.02)  # Added bearing noise
ekf_cfg = EKFConfig()

# Simulate both range and bearing measurements
poses_noisy, odometry_traj = simulate_vio_odometry(trajectory, vio_cfg)

# Run EKF with range and bearing fusion
state_seq_rb, cov_seq_rb, t_seq_rb = simulate_ekf_with_bearing(
    trajectory,
    anchor_pos,
    ekf_cfg,
    (vio_cfg, uwb_cfg),
    seed=42
)

# Convert to arrays for plotting
poses_est_rb = np.array(state_seq_rb)

# Plot results and compare with range-only
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: Range+bearing estimate
plot_trajectory_2d(ax1, poses_gt, label='Ground Truth', color='k')
plot_trajectory_2d(ax1, poses_est_rb, label='EKF (Range+Bearing)', color='g', linestyle='--')
plot_anchor_positions(ax1, np.array([anchor_pos]))

for i in range(0, len(poses_est_rb), plot_every):
    plot_error_ellipse(ax1, poses_est_rb[i], cov_seq_rb[i], color='green', n_std=2.0)

ax1.set_title('Range+Bearing EKF')
ax1.legend()

# Right: Compare uncertainty
fig2, ax3 = plt.subplots(figsize=(8, 6))
plot_trajectory_2d(ax3, poses_gt, label='Ground Truth', color='k')
ax3.set_title('Comparison of EKF Estimates')
ax3.legend()

plt.tight_layout()
plt.show()